# Creating a research agent with parallel nodes
Companion notebook for the **Creating a research agent with parallel nodes** section of Chapter 4.

This is version 2 of the research agent: `search_sources` (Wikipedia) and `search_from_tavily` (web search) now run **in parallel**, and the state uses reducers so their results merge safely instead of one overwriting the other.

> Add `TAVILY_API_KEY` (and `OPENAI_API_KEY`) to your `.env` to see real web results and a real summary. Without them the notebook still completes -- the Tavily node records an error and moves on, exactly as the book's `search_from_tavily` function does.

In [1]:
import os
import operator
from typing import Annotated
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()


def keep_non_empty(a: str, b: str) -> str:
    """Reducer that keeps the non-empty string value."""
    return a if a else b


class ResearchState(BaseModel):
    """Main state for the research workflow, now with reducers for parallel execution."""

    query: Annotated[str, keep_non_empty] = Field(default="", description="The research question")
    candidate_urls: Annotated[list[str], operator.add] = Field(default_factory=list, description="Source URLs, merged")
    page_texts: Annotated[list[str], operator.add] = Field(default_factory=list, description="Fetched page content, merged")
    draft_summary: Annotated[str, keep_non_empty] = Field(default="", description="Preliminary summary")
    final_report: Annotated[str, keep_non_empty] = Field(default="", description="The finalized research report")
    errors: Annotated[list[str], operator.add] = Field(default_factory=list, description="Error messages, merged")

In [2]:
from langchain_community.retrievers import WikipediaRetriever
from langchain_openai import ChatOpenAI
from tavily import TavilyClient

wiki = WikipediaRetriever()
# Falls back to a dummy key so the client can be constructed even when OPENAI_API_KEY
# is not set; the guard inside summarize_results() stops it from ever being called.
llm = ChatOpenAI(model_name="gpt-4", temperature=0.0, api_key=os.getenv("OPENAI_API_KEY", "sk-not-set"))


def prepare_query(state: ResearchState) -> dict:
    return {"query": state.query.strip()}


def search_sources(state: ResearchState) -> dict:
    query = state.query
    if not query:
        return {"candidate_urls": [], "page_texts": []}
    try:
        docs = wiki.invoke(query)
    except Exception as e:
        return {"candidate_urls": [], "page_texts": [], "errors": [f"Wikipedia error: {e}"]}
    urls = [doc.metadata.get("url", doc.metadata.get("title", "")) for doc in docs]
    texts = [doc.page_content for doc in docs]
    return {"candidate_urls": urls, "page_texts": texts}

C:\Users\himan\AppData\Local\Temp\ipykernel_34120\1388625085.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


`search_from_tavily` runs the web search. It is the *second* parallel branch, alongside `search_sources`.

In [3]:
def search_from_tavily(state: ResearchState) -> dict:
    query = state.query
    if not query:
        return {"candidate_urls": [], "errors": []}

    web_urls = []
    errors = []
    tavily_api_key = os.getenv("TAVILY_API_KEY")

    try:
        if not tavily_api_key:
            raise ValueError("TAVILY_API_KEY not set in environment variables")
        tavily_client = TavilyClient(api_key=tavily_api_key)
        response = tavily_client.search(query=query, max_results=5)
        web_urls = [result["url"] for result in response["results"]]
    except Exception as e:
        errors.append(f"Tavily search error: {e}")

    return {"candidate_urls": web_urls, "errors": errors}

In [4]:
def summarize_results(state: ResearchState) -> dict:
    combined = "\n\n".join(state.page_texts)
    if not combined:
        return {"final_report": "No content found."}
    if not os.getenv("OPENAI_API_KEY"):
        return {"final_report": "[Skipped LLM call: set OPENAI_API_KEY in your .env to generate a real summary]"}
    prompt = f"Summarize the key points and cite sources if needed:\n\n{combined}"
    try:
        summary = llm.invoke(prompt)
        return {"final_report": summary.content}
    except Exception as e:
        return {"final_report": f"[LLM call failed: {e}]"}

`prepare_query` fans out into both search nodes; both flow into `summarize_results`.

In [5]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(ResearchState)
workflow.add_node("prepare_query", prepare_query)
workflow.add_node("search_sources", search_sources)
workflow.add_node("search_from_tavily", search_from_tavily)
workflow.add_node("summarize_results", summarize_results)

workflow.add_edge(START, "prepare_query")

workflow.add_edge("prepare_query", "search_sources")
workflow.add_edge("prepare_query", "search_from_tavily")

workflow.add_edge("search_sources", "summarize_results")
workflow.add_edge("search_from_tavily", "summarize_results")

workflow.add_edge("summarize_results", END)

app = workflow.compile()

In [6]:
initial_state = {
    "query": "quantum computing applications",
    "candidate_urls": [],
    "page_texts": [],
    "draft_summary": "",
    "final_report": "",
    "errors": [],
}

final_state = app.invoke(initial_state)

print(f"URLs found (Wikipedia + Tavily merged): {len(final_state['candidate_urls'])}")
for url in final_state["candidate_urls"]:
    print(f"  - {url}")
if final_state["errors"]:
    print("\nErrors:")
    for err in final_state["errors"]:
        print(f"  - {err}")
print(f"\n{final_state['final_report']}")

URLs found (Wikipedia + Tavily merged): 3
  - Cloud-based quantum computing
  - Quantum computing
  - Quantinuum

Errors:
  - Tavily search error: TAVILY_API_KEY not set in environment variables

[Skipped LLM call: set OPENAI_API_KEY in your .env to generate a real summary]
